# MT v3: multi-run training + weighted voting ensemble for Chinese → Vietnamese

This v3 notebook is designed specifically to improve OJ accuracy by going beyond a single model.

What changed from v2:

- trains **multiple runs** with different **seeds / tokenizers / model sizes**
- keeps the strong parts from v2: **Transformer**, **SentencePiece**, **label smoothing**, **AMP**, **grad clipping**, **warmup + cosine**, **checkpoint averaging**
- adds a cleaner **parallel-corpus cleaning** stage
- uses a **monitor split** only to choose settings and estimate weights
- optionally **re-trains each selected experiment on the full training set**
- combines models with **weighted MBR / consensus voting**
- optionally uses a tiny **target-side Vietnamese n-gram LM** (trained only on `train.vi`) to break ties in reranking
- exports final **submission.csv** and **submission.zip**

Contest-safe assumptions:

- **No pretrained translation models**
- tokenizers are trained **from scratch** on the provided data only
- all models are trained **from scratch**


In [1]:
import importlib
import subprocess
import sys

def ensure_package(module_name, pip_name=None):
    pip_name = pip_name or module_name
    try:
        importlib.import_module(module_name)
    except ImportError:
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

ensure_package("sentencepiece")
ensure_package("sacrebleu")


In [2]:
import gc
import json
import math
import os
import random
import re
import shutil
import time
import unicodedata
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import sacrebleu
import sentencepiece as spm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = torch.cuda.is_available()

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

print("DEVICE:", DEVICE)
print("AMP   :", AMP_ENABLED)


DEVICE: cuda
AMP   : True


/home/izu/Projects/.venv/.venv-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

There are two levels of config:

1. **Global notebook config**: paths, split size, ensemble settings
2. **Experiment presets**: each one trains a different model family

The default list below is a solid starting point for strict OJ scoring:

- `bpe_base_s42`
- `unigram_base_s67`
- `bpe_wide_s2024`

If GPU time is limited, keep only the first **2** experiments.


In [3]:
@dataclass
class GlobalCFG:
    # paths
    data_dir: str = "dataset"
    work_dir: str = "mt_v3_artifacts"

    # data / split
    val_ratio: float = 0.04
    split_seed: int = 2024
    deduplicate_exact_pairs: bool = True
    max_char_len_src: int = 220
    max_char_len_tgt: int = 260
    min_char_len_src: int = 1
    min_char_len_tgt: int = 1
    max_len_ratio: float = 6.0
    min_len_ratio: float = 0.10

    # training / runtime
    num_workers: int = 2
    pin_memory: bool = True
    train_shuffle: bool = True
    force_retrain_tokenizer: bool = False
    force_retrain_models: bool = False

    # stage B: retrain on full cleaned train set after picking best epoch from monitor split
    do_full_retrain: bool = True
    full_retrain_extra_epochs: int = 1
    full_retrain_average_last_k: int = 3

    # ensemble / reranking
    keep_top_models_for_ensemble: int = 3
    nbest_per_model: int = 2
    use_target_lm_rerank: bool = True
    ensemble_consensus_weight: float = 0.72
    ensemble_presence_weight: float = 0.18
    ensemble_lm_weight: float = 0.10

    # decode preview
    preview_valid_samples: int = 120

GLOBAL_CFG = GlobalCFG()

@dataclass
class ExperimentPreset:
    name: str
    seed: int
    sp_model_type: str = "bpe"            # "bpe" or "unigram"
    src_vocab_size: int = 4092
    tgt_vocab_size: int = 4092
    character_coverage_zh: float = 0.9995

    max_src_len: int = 180
    max_tgt_len: int = 200

    d_model: int = 384
    nhead: int = 6
    num_encoder_layers: int = 5
    num_decoder_layers: int = 5
    dim_feedforward: int = 1536
    dropout: float = 0.15

    epochs: int = 18
    batch_size: int = 80
    grad_accum_steps: int = 2
    lr: float = 3e-4
    weight_decay: float = 1e-4
    warmup_ratio: float = 0.08
    label_smoothing: float = 0.1
    grad_clip: float = 1.0

    beam_size: int = 4
    length_penalty_alpha: float = 0.7
    no_repeat_ngram_size: int = 3
    max_decode_len: int = 200
    save_top_k: int = 3

EXPERIMENTS = [
    ExperimentPreset(
        name="bpe_base_s42",
        seed=42,
        sp_model_type="bpe",
        d_model=384,
        nhead=6,
        num_encoder_layers=5,
        num_decoder_layers=5,
        dim_feedforward=1536,
        dropout=0.15,
        batch_size=80,
        grad_accum_steps=2,
        beam_size=4,
    ),
    ExperimentPreset(
        name="unigram_base_s67",
        seed=67,
        sp_model_type="unigram",
        d_model=384,
        nhead=6,
        num_encoder_layers=5,
        num_decoder_layers=5,
        dim_feedforward=1536,
        dropout=0.15,
        batch_size=80,
        grad_accum_steps=2,
        beam_size=4,
    ),
    ExperimentPreset(
        name="bpe_wide_s2024",
        seed=2024,
        sp_model_type="bpe",
        d_model=512,
        nhead=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        dim_feedforward=2048,
        dropout=0.15,
        batch_size=48,          # wider model => smaller batch
        grad_accum_steps=3,
        beam_size=5,
        length_penalty_alpha=0.75,
    ),
]

print(json.dumps(asdict(GLOBAL_CFG), indent=2))
print("Experiments:")
for exp in EXPERIMENTS:
    print(" -", exp.name, "|", exp.sp_model_type, "| d_model =", exp.d_model, "| seed =", exp.seed)


{
  "data_dir": "dataset",
  "work_dir": "mt_v3_artifacts",
  "val_ratio": 0.04,
  "split_seed": 2024,
  "deduplicate_exact_pairs": true,
  "max_char_len_src": 220,
  "max_char_len_tgt": 260,
  "min_char_len_src": 1,
  "min_char_len_tgt": 1,
  "max_len_ratio": 6.0,
  "min_len_ratio": 0.1,
  "num_workers": 2,
  "pin_memory": true,
  "train_shuffle": true,
  "force_retrain_tokenizer": false,
  "force_retrain_models": false,
  "do_full_retrain": true,
  "full_retrain_extra_epochs": 1,
  "full_retrain_average_last_k": 3,
  "keep_top_models_for_ensemble": 3,
  "nbest_per_model": 2,
  "use_target_lm_rerank": true,
  "ensemble_consensus_weight": 0.72,
  "ensemble_presence_weight": 0.18,
  "ensemble_lm_weight": 0.1,
  "preview_valid_samples": 120
}
Experiments:
 - bpe_base_s42 | bpe | d_model = 384 | seed = 42
 - unigram_base_s67 | unigram | d_model = 384 | seed = 67
 - bpe_wide_s2024 | bpe | d_model = 512 | seed = 2024


## Data utilities and cleaning

This stage does a little more than v2:

- Unicode normalization
- whitespace / punctuation cleanup
- optional exact-pair deduplication
- simple outlier filtering by source / target length
- keeps raw `test.zh` untouched for final CSV, but normalizes a clean copy for inference


In [4]:
def find_dataset_dir(preferred: str = "dataset") -> Path:
    candidates = [
        Path(preferred),
        Path("/content/dataset"),
        Path("./dataset"),
        Path("../dataset"),
        Path("/kaggle/input/dataset/dataset"),
    ]
    for path in candidates:
        if (path / "train" / "train.zh").exists() and (path / "train" / "train.vi").exists() and (path / "test" / "test.zh").exists():
            return path.resolve()
    raise FileNotFoundError(
        "Could not find dataset/. Expected train/train.zh, train/train.vi, and test/test.zh. "
        "Set GlobalCFG.data_dir to the correct folder."
    )

def read_lines(path: Path) -> List[str]:
    with open(path, "r", encoding="utf-8") as f:
        return [line.rstrip("\n") for line in f]

def write_lines(path: Path, lines: Iterable[str]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(f"{line}\n")

def normalize_zh(text: str) -> str:
    text = unicodedata.normalize("NFKC", text.strip())
    text = text.replace("\u3000", " ")
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    text = text.replace("…", "...")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def normalize_vi(text: str) -> str:
    text = unicodedata.normalize("NFKC", text.strip())
    text = text.replace("\u3000", " ")
    text = text.replace("“", '"').replace("”", '"').replace("‘", "'").replace("’", "'")
    text = text.replace("…", "...")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:!?%])", r"\1", text)
    text = re.sub(r"([(])\s+", r"\1", text)
    text = re.sub(r"\s+([)])", r"\1", text)
    text = re.sub(r"\s+([/])\s*", r"\1", text)
    return text.strip()

def postprocess_vi(text: str) -> str:
    text = unicodedata.normalize("NFKC", text.strip())
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s+([,.;:!?%])", r"\1", text)
    text = re.sub(r"([(])\s+", r"\1", text)
    text = re.sub(r"\s+([)])", r"\1", text)
    text = re.sub(r"\s+([/])\s*", r"\1", text)
    text = re.sub(r'\s+"', ' "', text)
    text = re.sub(r'"\s+', '" ', text)
    text = re.sub(r"\s+'", " '", text)
    text = re.sub(r"'\s+", "' ", text)
    return text.strip()

def should_copy_source(src: str) -> bool:
    s = src.strip()
    if not s:
        return True
    if re.search(r"(https?://|www\.|@)", s):
        return True
    if not re.search(r"[\u3400-\u4dbf\u4e00-\u9fff]", s):
        return True
    return False

def clean_parallel_corpus(src_lines: List[str], tgt_lines: List[str], cfg: GlobalCFG):
    assert len(src_lines) == len(tgt_lines)
    cleaned = []
    seen = set()
    dropped = Counter()

    for raw_src, raw_tgt in zip(src_lines, tgt_lines):
        src = normalize_zh(raw_src)
        tgt = normalize_vi(raw_tgt)

        if not src or not tgt:
            dropped["empty"] += 1
            continue

        src_len = max(1, len(src))
        tgt_len = max(1, len(tgt))
        ratio = tgt_len / src_len

        if src_len < cfg.min_char_len_src or tgt_len < cfg.min_char_len_tgt:
            dropped["too_short"] += 1
            continue
        if src_len > cfg.max_char_len_src or tgt_len > cfg.max_char_len_tgt:
            dropped["too_long"] += 1
            continue
        if ratio < cfg.min_len_ratio or ratio > cfg.max_len_ratio:
            dropped["bad_ratio"] += 1
            continue

        pair = (src, tgt)
        if cfg.deduplicate_exact_pairs:
            if pair in seen:
                dropped["duplicate_pair"] += 1
                continue
            seen.add(pair)

        cleaned.append(pair)

    clean_src = [x[0] for x in cleaned]
    clean_tgt = [x[1] for x in cleaned]
    return clean_src, clean_tgt, dropped

def make_train_val_split(src_lines: List[str], tgt_lines: List[str], val_ratio: float, seed: int):
    assert len(src_lines) == len(tgt_lines)
    indices = list(range(len(src_lines)))
    rng = random.Random(seed)
    rng.shuffle(indices)

    val_size = max(1, int(len(indices) * val_ratio))
    val_idx = set(indices[:val_size])

    tr_src, tr_tgt, va_src, va_tgt = [], [], [], []
    for i, (s, t) in enumerate(zip(src_lines, tgt_lines)):
        if i in val_idx:
            va_src.append(s)
            va_tgt.append(t)
        else:
            tr_src.append(s)
            tr_tgt.append(t)
    return tr_src, tr_tgt, va_src, va_tgt


In [5]:
data_dir = find_dataset_dir(GLOBAL_CFG.data_dir)
work_dir = Path(GLOBAL_CFG.work_dir)
work_dir.mkdir(parents=True, exist_ok=True)

train_zh_path = data_dir / "train" / "train.zh"
train_vi_path = data_dir / "train" / "train.vi"
test_zh_path = data_dir / "test" / "test.zh"

raw_train_zh = read_lines(train_zh_path)
raw_train_vi = read_lines(train_vi_path)
raw_test_zh = read_lines(test_zh_path)

assert len(raw_train_zh) == len(raw_train_vi), "train.zh and train.vi must have identical line counts"

clean_train_zh, clean_train_vi, dropped_stats = clean_parallel_corpus(raw_train_zh, raw_train_vi, GLOBAL_CFG)
clean_test_zh = [normalize_zh(x) for x in raw_test_zh]

mon_tr_src, mon_tr_tgt, mon_va_src, mon_va_tgt = make_train_val_split(
    clean_train_zh,
    clean_train_vi,
    val_ratio=GLOBAL_CFG.val_ratio,
    seed=GLOBAL_CFG.split_seed,
)

print("dataset dir          :", data_dir)
print("raw train pairs      :", len(raw_train_zh))
print("clean train pairs    :", len(clean_train_zh))
print("dropped stats        :", dict(dropped_stats))
print("monitor train pairs  :", len(mon_tr_src))
print("monitor valid pairs  :", len(mon_va_src))
print("test lines           :", len(clean_test_zh))
print()
print("sample zh:", clean_train_zh[0] if clean_train_zh else "(empty)")
print("sample vi:", clean_train_vi[0] if clean_train_vi else "(empty)")


dataset dir          : /home/izu/Projects/olpai/trans/dataset
raw train pairs      : 25648
clean train pairs    : 24867
dropped stats        : {'duplicate_pair': 752, 'bad_ratio': 29}
monitor train pairs  : 23873
monitor valid pairs  : 994
test lines           : 6413

sample zh: 这个 巴士 去 联合 广场 的 度假 旅馆 吗 ?
sample vi: Xe_buýt này có đi đến Quảng_trường Holiday_Inn_Union không?


## Tokenizer helpers

Each experiment gets its **own tokenizer folder**, so BPE and unigram runs can coexist cleanly.


In [6]:
@dataclass
class SpecialTokens:
    pad_id: int
    unk_id: int
    bos_id: int
    eos_id: int

def train_or_load_tokenizers(exp: ExperimentPreset, full_src_lines: List[str], full_tgt_lines: List[str], root_dir: Path):
    exp_dir = root_dir / exp.name
    sp_dir = exp_dir / "spm"
    sp_dir.mkdir(parents=True, exist_ok=True)

    src_txt = sp_dir / "train_src.txt"
    tgt_txt = sp_dir / "train_tgt.txt"
    src_prefix = sp_dir / "src_sp"
    tgt_prefix = sp_dir / "tgt_sp"

    src_model = src_prefix.with_suffix(".model")
    tgt_model = tgt_prefix.with_suffix(".model")

    if GLOBAL_CFG.force_retrain_tokenizer or not src_model.exists() or not tgt_model.exists():
        write_lines(src_txt, full_src_lines)
        write_lines(tgt_txt, full_tgt_lines)

        spm.SentencePieceTrainer.train(
            input=str(src_txt),
            model_prefix=str(src_prefix),
            vocab_size=exp.src_vocab_size,
            model_type=exp.sp_model_type,
            character_coverage=exp.character_coverage_zh,
            pad_id=0, unk_id=1, bos_id=2, eos_id=3,
            pad_piece="<pad>", unk_piece="<unk>", bos_piece="<bos>", eos_piece="<eos>",
            split_digits=True,
            shuffle_input_sentence=True,
            input_sentence_size=min(len(full_src_lines), 1_000_000),
            train_extremely_large_corpus=False,
        )

        spm.SentencePieceTrainer.train(
            input=str(tgt_txt),
            model_prefix=str(tgt_prefix),
            vocab_size=exp.tgt_vocab_size,
            model_type=exp.sp_model_type,
            character_coverage=1.0,
            pad_id=0, unk_id=1, bos_id=2, eos_id=3,
            pad_piece="<pad>", unk_piece="<unk>", bos_piece="<bos>", eos_piece="<eos>",
            split_digits=True,
            shuffle_input_sentence=True,
            input_sentence_size=min(len(full_tgt_lines), 1_000_000),
            train_extremely_large_corpus=False,
        )

    sp_src = spm.SentencePieceProcessor(model_file=str(src_model))
    sp_tgt = spm.SentencePieceProcessor(model_file=str(tgt_model))

    special = SpecialTokens(
        pad_id=sp_src.pad_id(),
        unk_id=sp_src.unk_id(),
        bos_id=sp_src.bos_id(),
        eos_id=sp_src.eos_id(),
    )
    assert (special.pad_id, special.unk_id, special.bos_id, special.eos_id) == (0, 1, 2, 3)
    return sp_src, sp_tgt, special


## Dataset and dataloaders

In [7]:
class TranslationDataset(Dataset):
    def __init__(
        self,
        src_lines: List[str],
        tgt_lines: Optional[List[str]],
        sp_src,
        sp_tgt,
        max_src_len: int,
        max_tgt_len: int,
        bos_id: int,
        eos_id: int,
    ):
        self.src_lines = src_lines
        self.tgt_lines = tgt_lines
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.bos_id = bos_id
        self.eos_id = eos_id

    def __len__(self):
        return len(self.src_lines)

    def encode_src(self, text: str) -> List[int]:
        ids = self.sp_src.encode(text, out_type=int)[: self.max_src_len - 2]
        return [self.bos_id] + ids + [self.eos_id]

    def encode_tgt(self, text: str) -> List[int]:
        ids = self.sp_tgt.encode(text, out_type=int)[: self.max_tgt_len - 2]
        return [self.bos_id] + ids + [self.eos_id]

    def __getitem__(self, idx: int):
        item = {
            "src_ids": self.encode_src(self.src_lines[idx]),
            "src_text": self.src_lines[idx],
        }
        if self.tgt_lines is not None:
            item["tgt_ids"] = self.encode_tgt(self.tgt_lines[idx])
            item["tgt_text"] = self.tgt_lines[idx]
        return item

def pad_sequences(seqs: List[List[int]], pad_value: int) -> torch.Tensor:
    max_len = max(len(x) for x in seqs)
    out = torch.full((len(seqs), max_len), pad_value, dtype=torch.long)
    for i, seq in enumerate(seqs):
        out[i, : len(seq)] = torch.tensor(seq, dtype=torch.long)
    return out

def build_collate_fn(pad_id: int):
    def collate_fn(batch: List[Dict]):
        batch = sorted(batch, key=lambda x: len(x["src_ids"]), reverse=True)
        src = pad_sequences([x["src_ids"] for x in batch], pad_id)
        out = {"src": src, "src_text": [x["src_text"] for x in batch]}
        if "tgt_ids" in batch[0]:
            tgt = pad_sequences([x["tgt_ids"] for x in batch], pad_id)
            out["tgt"] = tgt
            out["tgt_text"] = [x["tgt_text"] for x in batch]
        return out
    return collate_fn

def make_loader(src_lines, tgt_lines, exp: ExperimentPreset, sp_src, sp_tgt, special: SpecialTokens, shuffle: bool):
    ds = TranslationDataset(
        src_lines=src_lines,
        tgt_lines=tgt_lines,
        sp_src=sp_src,
        sp_tgt=sp_tgt,
        max_src_len=exp.max_src_len,
        max_tgt_len=exp.max_tgt_len,
        bos_id=special.bos_id,
        eos_id=special.eos_id,
    )
    loader = DataLoader(
        ds,
        batch_size=exp.batch_size,
        shuffle=shuffle,
        num_workers=GLOBAL_CFG.num_workers,
        pin_memory=GLOBAL_CFG.pin_memory,
        collate_fn=build_collate_fn(special.pad_id),
    )
    return ds, loader


## Transformer model

Still trained entirely from scratch, but slightly stronger than the v2 baseline:

- `activation="gelu"`
- tied target embedding / generator
- pre-LN Transformer (`norm_first=True`)


In [8]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.pe[:, : x.size(1)]

class TransformerMT(nn.Module):
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int,
        nhead: int,
        num_encoder_layers: int,
        num_decoder_layers: int,
        dim_feedforward: int,
        dropout: float,
        pad_id: int,
    ):
        super().__init__()
        self.pad_id = pad_id
        self.d_model = d_model

        self.src_emb = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_id)
        self.tgt_emb = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model)
        self.dropout = nn.Dropout(dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )

        self.generator = nn.Linear(d_model, tgt_vocab_size, bias=False)
        self.generator.weight = self.tgt_emb.weight
        self._reset_parameters()

    def _reset_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def src_padding_mask(self, src: torch.Tensor) -> torch.Tensor:
        return src.eq(self.pad_id)

    def tgt_padding_mask(self, tgt: torch.Tensor) -> torch.Tensor:
        return tgt.eq(self.pad_id)

    def causal_mask(self, size: int, device: torch.device) -> torch.Tensor:
        return torch.triu(torch.full((size, size), float("-inf"), device=device), diagonal=1)

    def encode(self, src: torch.Tensor):
        src_pad = self.src_padding_mask(src)
        src_x = self.dropout(self.pos(self.src_emb(src) * math.sqrt(self.d_model)))
        memory = self.transformer.encoder(src_x, src_key_padding_mask=src_pad)
        return memory, src_pad

    def decode(self, tgt: torch.Tensor, memory: torch.Tensor, src_pad: torch.Tensor):
        tgt_pad = self.tgt_padding_mask(tgt)
        tgt_mask = self.causal_mask(tgt.size(1), tgt.device)
        tgt_x = self.dropout(self.pos(self.tgt_emb(tgt) * math.sqrt(self.d_model)))
        out = self.transformer.decoder(
            tgt=tgt_x,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_pad,
            memory_key_padding_mask=src_pad,
        )
        return self.generator(out)

    def forward(self, src: torch.Tensor, tgt_in: torch.Tensor):
        memory, src_pad = self.encode(src)
        return self.decode(tgt_in, memory, src_pad)

def build_model(exp: ExperimentPreset, src_vocab_size: int, tgt_vocab_size: int, pad_id: int):
    model = TransformerMT(
        src_vocab_size=src_vocab_size,
        tgt_vocab_size=tgt_vocab_size,
        d_model=exp.d_model,
        nhead=exp.nhead,
        num_encoder_layers=exp.num_encoder_layers,
        num_decoder_layers=exp.num_decoder_layers,
        dim_feedforward=exp.dim_feedforward,
        dropout=exp.dropout,
        pad_id=pad_id,
    )
    return model.to(DEVICE)

def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Training helpers

In [9]:
def shift_tgt_for_teacher_forcing(tgt: torch.Tensor):
    return tgt[:, :-1], tgt[:, 1:]

def compute_loss(logits: torch.Tensor, tgt_out: torch.Tensor, pad_id: int, label_smoothing: float):
    vocab_size = logits.size(-1)
    return F.cross_entropy(
        logits.reshape(-1, vocab_size),
        tgt_out.reshape(-1),
        ignore_index=pad_id,
        label_smoothing=label_smoothing,
    )

def build_optimizer_and_scheduler(model: nn.Module, exp: ExperimentPreset, total_steps: int):
    optimizer = AdamW(
        model.parameters(),
        lr=exp.lr,
        betas=(0.9, 0.98),
        eps=1e-9,
        weight_decay=exp.weight_decay,
    )
    warmup_steps = max(1, int(total_steps * exp.warmup_ratio))

    def lr_lambda(step_idx: int):
        step = step_idx + 1
        if step <= warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.1, 0.5 * (1.0 + math.cos(math.pi * progress)))

    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)
    return optimizer, scheduler

def save_checkpoint(path: Path, model: nn.Module, optimizer, scheduler, epoch: int, metrics: Dict, extra: Optional[Dict] = None):
    state = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict() if optimizer is not None else None,
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "epoch": epoch,
        "metrics": metrics,
        "extra": extra or {},
    }
    torch.save(state, path)

def load_model_state(model: nn.Module, ckpt_path: Path):
    state = torch.load(ckpt_path, map_location=DEVICE)
    if "model_state" in state:
        model.load_state_dict(state["model_state"], strict=True)
    else:
        model.load_state_dict(state, strict=True)
    return model

def get_scored_checkpoints(ckpt_dir: Path, key_name: str = "bleu") -> List[Tuple[float, Path]]:
    scored = []
    for p in ckpt_dir.glob("*.pt"):
        m = re.search(rf"{key_name}([0-9]+(?:\.[0-9]+)?)", p.stem)
        if m:
            scored.append((float(m.group(1)), p))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored

def average_checkpoints(ckpt_paths: List[Path], out_path: Path):
    assert ckpt_paths, "No checkpoints to average"
    avg_state = None
    count = 0
    for p in ckpt_paths:
        state = torch.load(p, map_location="cpu")
        model_state = state["model_state"] if "model_state" in state else state
        if avg_state is None:
            avg_state = {k: v.detach().clone().float() for k, v in model_state.items()}
        else:
            for k in avg_state:
                avg_state[k] += model_state[k].detach().float()
        count += 1
    for k in avg_state:
        avg_state[k] /= count
    torch.save({"model_state": avg_state, "paths": [str(x) for x in ckpt_paths]}, out_path)
    return out_path

def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Decoding helpers

Important additions for v3:

- `beam_search_decode_nbest(...)` returns **top-n candidates**
- later, the ensemble picks the final answer using **weighted consensus voting**


In [10]:
def _has_repeat_ngram(tokens: List[int], cand: int, n: int) -> bool:
    if n <= 0 or len(tokens) + 1 < n:
        return False
    trial = tokens + [cand]
    target_ngram = tuple(trial[-n:])
    for i in range(len(trial) - n):
        if tuple(trial[i : i + n]) == target_ngram:
            return True
    return False

def _normalized_beam_score(log_prob_sum: float, length: int, alpha: float) -> float:
    length = max(length, 1)
    return log_prob_sum / (((5 + length) / 6) ** alpha)

@torch.no_grad()
def beam_search_decode_nbest(
    model: TransformerMT,
    src_text: str,
    sp_src,
    sp_tgt,
    special: SpecialTokens,
    exp: ExperimentPreset,
    nbest: int = 1,
) -> List[str]:
    if should_copy_source(src_text):
        return [src_text]

    src_ids = [special.bos_id] + sp_src.encode(src_text, out_type=int)[: exp.max_src_len - 2] + [special.eos_id]
    src = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)
    memory, src_pad = model.encode(src)

    beams = [([special.bos_id], 0.0, False)]
    finished = []

    for _ in range(exp.max_decode_len):
        candidates = []
        for tokens, score, done in beams:
            if done:
                candidates.append((tokens, score, done))
                continue

            tgt = torch.tensor(tokens, dtype=torch.long, device=DEVICE).unsqueeze(0)
            logits = model.decode(tgt, memory, src_pad)[0, -1]
            log_probs = F.log_softmax(logits, dim=-1)

            log_probs[special.unk_id] = -1e9
            if len(tokens) < 2:
                log_probs[special.eos_id] = -1e9

            topk_scores, topk_ids = torch.topk(log_probs, k=max(exp.beam_size * 2, nbest * 2))
            for lp, token_id in zip(topk_scores.tolist(), topk_ids.tolist()):
                if exp.no_repeat_ngram_size > 0 and _has_repeat_ngram(tokens, token_id, exp.no_repeat_ngram_size):
                    continue
                new_tokens = tokens + [token_id]
                new_score = score + lp
                is_done = token_id == special.eos_id
                candidates.append((new_tokens, new_score, is_done))

        candidates.sort(key=lambda x: _normalized_beam_score(x[1], len(x[0]), exp.length_penalty_alpha), reverse=True)
        beams = candidates[: max(exp.beam_size, nbest)]
        finished.extend([x for x in beams if x[2]])

        if len(finished) >= max(exp.beam_size, nbest):
            best_finished = max(finished, key=lambda x: _normalized_beam_score(x[1], len(x[0]), exp.length_penalty_alpha))
            best_active = max(beams, key=lambda x: _normalized_beam_score(x[1], len(x[0]), exp.length_penalty_alpha))
            if _normalized_beam_score(best_finished[1], len(best_finished[0]), exp.length_penalty_alpha) >= _normalized_beam_score(best_active[1], len(best_active[0]), exp.length_penalty_alpha):
                break

    pool = finished if finished else beams
    pool.sort(key=lambda x: _normalized_beam_score(x[1], len(x[0]), exp.length_penalty_alpha), reverse=True)

    uniq = []
    seen = set()
    for tokens, score, done in pool:
        body = tokens[1:]
        if special.eos_id in body:
            body = body[: body.index(special.eos_id)]
        text = postprocess_vi(sp_tgt.decode(body))
        if text and text not in seen:
            seen.add(text)
            uniq.append(text)
        if len(uniq) >= nbest:
            break

    if not uniq:
        return [""]
    return uniq

@torch.no_grad()
def predict_texts(
    model: TransformerMT,
    src_texts: List[str],
    sp_src,
    sp_tgt,
    special: SpecialTokens,
    exp: ExperimentPreset,
    nbest: int = 1,
    desc: str = "decode",
) -> List:
    model.eval()
    outputs = []
    for src_text in tqdm(src_texts, desc=desc):
        cands = beam_search_decode_nbest(
            model=model,
            src_text=src_text,
            sp_src=sp_src,
            sp_tgt=sp_tgt,
            special=special,
            exp=exp,
            nbest=nbest,
        )
        outputs.append(cands if nbest > 1 else cands[0])
    return outputs

def corpus_bleu_score(preds: List[str], refs: List[str]) -> float:
    return sacrebleu.corpus_bleu(preds, [refs]).score

def corpus_chrf_score(preds: List[str], refs: List[str]) -> float:
    return sacrebleu.corpus_chrf(preds, [refs]).score


## Tiny target-side Vietnamese n-gram LM for reranking

This is **not** a pretrained model. It is trained only on `train.vi` and used only as a small tie-breaker during ensemble reranking.


In [11]:
def vi_lm_tokenize(text: str) -> List[str]:
    text = postprocess_vi(text.lower())
    text = re.sub(r"([,.;:!?()/%\"'])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split() if text else []

class SimpleWordNGramLM:
    def __init__(self, order: int = 3, add_k: float = 0.2):
        self.order = order
        self.add_k = add_k
        self.ngram_counts = [Counter() for _ in range(order)]
        self.context_counts = [Counter() for _ in range(order)]
        self.vocab = set()

    def fit(self, lines: List[str]):
        for line in tqdm(lines, desc="fit target LM"):
            toks = ["<s>"] * (self.order - 1) + vi_lm_tokenize(line) + ["</s>"]
            self.vocab.update(toks)
            for n in range(1, self.order + 1):
                for i in range(len(toks) - n + 1):
                    ngram = tuple(toks[i : i + n])
                    context = tuple(toks[i : i + n - 1])
                    self.ngram_counts[n - 1][ngram] += 1
                    self.context_counts[n - 1][context] += 1
        return self

    def score(self, text: str) -> float:
        toks = ["<s>"] * (self.order - 1) + vi_lm_tokenize(text) + ["</s>"]
        if len(toks) <= self.order - 1:
            return -999.0

        vocab_size = max(10, len(self.vocab))
        total_logp = 0.0
        total_steps = 0

        for idx in range(self.order - 1, len(toks)):
            total_steps += 1
            token = toks[idx]

            scored = False
            for n in range(self.order, 0, -1):
                start = idx - n + 1
                if start < 0:
                    continue
                ngram = tuple(toks[start : idx + 1])
                context = tuple(toks[start : idx])

                count = self.ngram_counts[n - 1][ngram]
                context_count = self.context_counts[n - 1][context]

                if count > 0 or n == 1:
                    prob = (count + self.add_k) / (context_count + self.add_k * vocab_size)
                    total_logp += math.log(prob)
                    scored = True
                    break

            if not scored:
                total_logp += math.log(1.0 / vocab_size)

        return total_logp / max(1, total_steps)


## Train / validate one experiment, then optionally retrain on the full train set

Workflow per experiment:

1. **Monitor stage**: train on train-split, evaluate on monitor-valid
2. take the best epoch / top checkpoints
3. **Full retrain stage**: train again on **all cleaned train pairs**
4. average the last few full-train checkpoints for final inference


In [12]:
def train_one_epoch(
    model: TransformerMT,
    loader: DataLoader,
    optimizer,
    scheduler,
    scaler,
    exp: ExperimentPreset,
    pad_id: int,
    epoch_idx: int,
):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, desc=f"train epoch {epoch_idx}", leave=False)
    for step, batch in enumerate(pbar, start=1):
        src = batch["src"].to(DEVICE, non_blocking=True)
        tgt = batch["tgt"].to(DEVICE, non_blocking=True)
        tgt_in, tgt_out = shift_tgt_for_teacher_forcing(tgt)

        with autocast(enabled=AMP_ENABLED):
            logits = model(src, tgt_in)
            loss = compute_loss(logits, tgt_out, pad_id=pad_id, label_smoothing=exp.label_smoothing)
            loss = loss / exp.grad_accum_steps

        scaler.scale(loss).backward()

        if step % exp.grad_accum_steps == 0 or step == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), exp.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

        running_loss += loss.item() * exp.grad_accum_steps
        avg_loss = running_loss / step
        pbar.set_postfix(loss=f"{avg_loss:.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    return running_loss / max(1, len(loader))

@torch.no_grad()
def evaluate_on_valid(
    model: TransformerMT,
    valid_src: List[str],
    valid_tgt: List[str],
    sp_src,
    sp_tgt,
    special: SpecialTokens,
    exp: ExperimentPreset,
):
    preds = predict_texts(
        model=model,
        src_texts=valid_src,
        sp_src=sp_src,
        sp_tgt=sp_tgt,
        special=special,
        exp=exp,
        nbest=1,
        desc=f"{exp.name} valid decode",
    )
    bleu = corpus_bleu_score(preds, valid_tgt)
    chrf = corpus_chrf_score(preds, valid_tgt)
    return preds, bleu, chrf

def fit_monitor_stage(
    exp: ExperimentPreset,
    exp_dir: Path,
    sp_src,
    sp_tgt,
    special: SpecialTokens,
    train_src: List[str],
    train_tgt: List[str],
    valid_src: List[str],
    valid_tgt: List[str],
):
    monitor_dir = exp_dir / "monitor_stage"
    ckpt_dir = monitor_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    meta_path = monitor_dir / "meta.json"

    if meta_path.exists() and not GLOBAL_CFG.force_retrain_models:
        with open(meta_path, "r", encoding="utf-8") as f:
            return json.load(f)

    seed_everything(exp.seed)

    _, train_loader = make_loader(train_src, train_tgt, exp, sp_src, sp_tgt, special, shuffle=GLOBAL_CFG.train_shuffle)
    model = build_model(exp, sp_src.vocab_size(), sp_tgt.vocab_size(), special.pad_id)
    print(f"[{exp.name}] trainable params: {count_parameters(model):,}")

    total_updates = math.ceil(len(train_loader) / exp.grad_accum_steps) * exp.epochs
    optimizer, scheduler = build_optimizer_and_scheduler(model, exp, total_updates)
    scaler = GradScaler(enabled=AMP_ENABLED)

    history = []
    best_bleu = -1.0

    for epoch in range(1, exp.epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            exp=exp,
            pad_id=special.pad_id,
            epoch_idx=epoch,
        )

        valid_preds, bleu, chrf = evaluate_on_valid(model, valid_src, valid_tgt, sp_src, sp_tgt, special, exp)
        ckpt_path = ckpt_dir / f"epoch{epoch:02d}_bleu{bleu:.4f}_chrf{chrf:.4f}.pt"
        save_checkpoint(
            ckpt_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            metrics={"bleu": bleu, "chrf": chrf, "train_loss": train_loss},
        )

        ranked = get_scored_checkpoints(ckpt_dir, key_name="bleu")
        for _, extra_path in ranked[exp.save_top_k:]:
            if extra_path.exists():
                extra_path.unlink()

        best_bleu = max(best_bleu, bleu)
        history.append({"epoch": epoch, "train_loss": train_loss, "valid_bleu": bleu, "valid_chrf": chrf, "best_bleu": best_bleu})
        print(history[-1])

    ranked = get_scored_checkpoints(ckpt_dir, key_name="bleu")
    top_ckpts = [p for _, p in ranked[: exp.save_top_k]]
    avg_ckpt = monitor_dir / "averaged_topk.pt"
    average_checkpoints(top_ckpts, avg_ckpt)

    best_epoch = int(pd.DataFrame(history).sort_values(["valid_bleu", "valid_chrf"], ascending=False).iloc[0]["epoch"])

    result = {
        "history": history,
        "best_epoch": best_epoch,
        "best_valid_bleu": float(max(x["valid_bleu"] for x in history)),
        "best_valid_chrf": float(max(x["valid_chrf"] for x in history)),
        "monitor_avg_ckpt": str(avg_ckpt),
        "monitor_top_ckpts": [str(p) for p in top_ckpts],
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    cleanup_cuda()
    return result

def fit_full_retrain_stage(
    exp: ExperimentPreset,
    exp_dir: Path,
    sp_src,
    sp_tgt,
    special: SpecialTokens,
    full_src: List[str],
    full_tgt: List[str],
    chosen_epochs: int,
):
    full_dir = exp_dir / "full_retrain_stage"
    ckpt_dir = full_dir / "checkpoints"
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    meta_path = full_dir / "meta.json"

    final_epochs = max(1, chosen_epochs + GLOBAL_CFG.full_retrain_extra_epochs)

    if meta_path.exists() and not GLOBAL_CFG.force_retrain_models:
        with open(meta_path, "r", encoding="utf-8") as f:
            return json.load(f)

    seed_everything(exp.seed + 1000)

    _, train_loader = make_loader(full_src, full_tgt, exp, sp_src, sp_tgt, special, shuffle=GLOBAL_CFG.train_shuffle)
    model = build_model(exp, sp_src.vocab_size(), sp_tgt.vocab_size(), special.pad_id)
    total_updates = math.ceil(len(train_loader) / exp.grad_accum_steps) * final_epochs
    optimizer, scheduler = build_optimizer_and_scheduler(model, exp, total_updates)
    scaler = GradScaler(enabled=AMP_ENABLED)

    history = []
    last_ckpts = []

    for epoch in range(1, final_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            exp=exp,
            pad_id=special.pad_id,
            epoch_idx=epoch,
        )

        ckpt_path = ckpt_dir / f"epoch{epoch:02d}_loss{train_loss:.4f}.pt"
        save_checkpoint(
            ckpt_path,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            epoch=epoch,
            metrics={"train_loss": train_loss},
        )
        history.append({"epoch": epoch, "train_loss": train_loss})
        last_ckpts.append(ckpt_path)

        if len(last_ckpts) > GLOBAL_CFG.full_retrain_average_last_k:
            old = last_ckpts.pop(0)
            if old.exists():
                old.unlink()

        print(history[-1])

    avg_ckpt = full_dir / "averaged_lastk.pt"
    average_checkpoints(last_ckpts, avg_ckpt)

    result = {
        "history": history,
        "final_epochs": final_epochs,
        "full_avg_ckpt": str(avg_ckpt),
        "full_last_ckpts": [str(p) for p in last_ckpts],
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    cleanup_cuda()
    return result

def run_experiment(exp: ExperimentPreset):
    print("=" * 80)
    print("Running experiment:", exp.name)
    print("=" * 80)

    exp_dir = work_dir / exp.name
    exp_dir.mkdir(parents=True, exist_ok=True)

    sp_src, sp_tgt, special = train_or_load_tokenizers(exp, clean_train_zh, clean_train_vi, work_dir)

    monitor_result = fit_monitor_stage(
        exp=exp,
        exp_dir=exp_dir,
        sp_src=sp_src,
        sp_tgt=sp_tgt,
        special=special,
        train_src=mon_tr_src,
        train_tgt=mon_tr_tgt,
        valid_src=mon_va_src,
        valid_tgt=mon_va_tgt,
    )

    if GLOBAL_CFG.do_full_retrain:
        full_result = fit_full_retrain_stage(
            exp=exp,
            exp_dir=exp_dir,
            sp_src=sp_src,
            sp_tgt=sp_tgt,
            special=special,
            full_src=clean_train_zh,
            full_tgt=clean_train_vi,
            chosen_epochs=int(monitor_result["best_epoch"]),
        )
        final_ckpt = full_result["full_avg_ckpt"]
        final_epochs = full_result["final_epochs"]
    else:
        full_result = None
        final_ckpt = monitor_result["monitor_avg_ckpt"]
        final_epochs = int(monitor_result["best_epoch"])

    artifact = {
        "name": exp.name,
        "seed": exp.seed,
        "sp_model_type": exp.sp_model_type,
        "src_vocab_size": exp.src_vocab_size,
        "tgt_vocab_size": exp.tgt_vocab_size,
        "pad_id": special.pad_id,
        "unk_id": special.unk_id,
        "bos_id": special.bos_id,
        "eos_id": special.eos_id,
        "monitor_best_epoch": int(monitor_result["best_epoch"]),
        "monitor_best_valid_bleu": float(monitor_result["best_valid_bleu"]),
        "monitor_best_valid_chrf": float(monitor_result["best_valid_chrf"]),
        "final_epochs_used": int(final_epochs),
        "final_ckpt": str(final_ckpt),
        "tokenizer_src_model": str((work_dir / exp.name / "spm" / "src_sp.model")),
        "tokenizer_tgt_model": str((work_dir / exp.name / "spm" / "tgt_sp.model")),
        "exp_cfg": asdict(exp),
        "monitor_result": monitor_result,
        "full_result": full_result,
    }

    artifact_path = exp_dir / "artifact.json"
    with open(artifact_path, "w", encoding="utf-8") as f:
        json.dump(artifact, f, ensure_ascii=False, indent=2)

    return artifact


## Train all experiments

This is the expensive part. If needed, reduce the experiment list in the config cell first.


In [13]:
all_artifacts = []
for exp in EXPERIMENTS:
    artifact = run_experiment(exp)
    all_artifacts.append(artifact)

results_df = pd.DataFrame(
    [
        {
            "name": a["name"],
            "seed": a["seed"],
            "sp_model_type": a["sp_model_type"],
            "monitor_best_epoch": a["monitor_best_epoch"],
            "monitor_best_valid_bleu": a["monitor_best_valid_bleu"],
            "monitor_best_valid_chrf": a["monitor_best_valid_chrf"],
            "final_epochs_used": a["final_epochs_used"],
            "final_ckpt": a["final_ckpt"],
        }
        for a in all_artifacts
    ]
).sort_values(["monitor_best_valid_bleu", "monitor_best_valid_chrf"], ascending=False).reset_index(drop=True)

results_df


Running experiment: bpe_base_s42


sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: mt_v3_artifacts/bpe_base_s42/spm/train_src.txt
  input_format: 
  model_prefix: mt_v3_artifacts/bpe_base_s42/spm/src_sp
  model_type: BPE
  vocab_size: 4092
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 24867
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 1
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <bos>
  eos_piece: <eos>
  pad_piece

[bpe_base_s42] trainable params: 23,849,472


/tmp/ipykernel_229589/2234840751.py:94: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=AMP_ENABLED)
train epoch 1:   0%|          | 0/299 [00:00<?, ?it/s]/tmp/ipykernel_229589/2234840751.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=AMP_ENABLED):
/home/izu/Projects/.venv/.venv-310/lib/python3.10/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
bpe_base_s42 valid decode: 100%|██████████| 994/994 [02:25<00:00,  6.84it/s]              


{'epoch': 1, 'train_loss': 6.394979791099012, 'valid_bleu': 3.2482630712450926, 'valid_chrf': 13.300948651628817, 'best_bleu': 3.2482630712450926}


...

{'epoch': 15, 'train_loss': 2.0260500911343304}


{'epoch': 16, 'train_loss': 2.007953976275604}


{'epoch': 17, 'train_loss': 1.994062254539115}


{'epoch': 18, 'train_loss': 1.978997062396452}


,name,seed,sp_model_type,monitor_best_epoch,monitor_best_valid_bleu,monitor_best_valid_chrf,final_epochs_used,final_ckpt
0,bpe_wide_s2024,2024,bpe,17,32.225273,46.238463,18,mt_v3_artifacts/bpe_wide_s2024/full_retrain_st...
1,bpe_base_s42,42,bpe,15,29.883411,43.875507,16,mt_v3_artifacts/bpe_base_s42/full_retrain_stag...
2,unigram_base_s67,67,unigram,18,29.678471,43.527513,19,mt_v3_artifacts/unigram_base_s67/full_retrain_...


## Load inference bundles and build the ensemble

In [14]:
@dataclass
class InferenceBundle:
    name: str
    weight: float
    model: nn.Module
    sp_src: object
    sp_tgt: object
    special: SpecialTokens
    exp: ExperimentPreset

def load_inference_bundle(artifact: Dict) -> InferenceBundle:
    exp = ExperimentPreset(**artifact["exp_cfg"])
    sp_src = spm.SentencePieceProcessor(model_file=artifact["tokenizer_src_model"])
    sp_tgt = spm.SentencePieceProcessor(model_file=artifact["tokenizer_tgt_model"])
    special = SpecialTokens(
        pad_id=artifact["pad_id"],
        unk_id=artifact["unk_id"],
        bos_id=artifact["bos_id"],
        eos_id=artifact["eos_id"],
    )
    model = build_model(exp, sp_src.vocab_size(), sp_tgt.vocab_size(), special.pad_id)
    load_model_state(model, Path(artifact["final_ckpt"]))
    model.eval()
    weight = max(0.1, float(artifact["monitor_best_valid_bleu"]))
    return InferenceBundle(
        name=artifact["name"],
        weight=weight,
        model=model,
        sp_src=sp_src,
        sp_tgt=sp_tgt,
        special=special,
        exp=exp,
    )

sorted_artifacts = sorted(all_artifacts, key=lambda x: x["monitor_best_valid_bleu"], reverse=True)
selected_artifacts = sorted_artifacts[: GLOBAL_CFG.keep_top_models_for_ensemble]
bundles = [load_inference_bundle(a) for a in selected_artifacts]

print("Selected models for ensemble:")
for b in bundles:
    print(f" - {b.name:20s} weight={b.weight:.4f}")


/home/izu/Projects/.venv/.venv-310/lib/python3.10/site-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


Selected models for ensemble:
 - bpe_wide_s2024       weight=32.2253
 - bpe_base_s42         weight=29.8834
 - unigram_base_s67     weight=29.6785


## Weighted MBR / consensus voting

How final selection works per source sentence:

1. each model produces **top-n candidates**
2. collect the union of candidates
3. score each candidate by:
   - similarity to other models' top predictions (**consensus**)
   - how many models produced it at all (**presence**)
   - optional small Vietnamese LM bonus (**fluency tiebreaker**)
4. choose the highest-scoring candidate


In [15]:
def sentence_chrf(a: str, b: str) -> float:
    return sacrebleu.sentence_chrf(a, [b]).score / 100.0

def minmax_scale(values: Dict[str, float]) -> Dict[str, float]:
    if not values:
        return {}
    vals = list(values.values())
    lo, hi = min(vals), max(vals)
    if abs(hi - lo) < 1e-12:
        return {k: 0.5 for k in values}
    return {k: (v - lo) / (hi - lo) for k, v in values.items()}

def choose_candidate_from_lists(
    candidate_lists: List[List[str]],
    weights: List[float],
    target_lm: Optional[SimpleWordNGramLM] = None,
) -> Tuple[str, List[Dict]]:
    assert len(candidate_lists) == len(weights)

    unique_candidates = []
    seen = set()
    for lst in candidate_lists:
        for cand in lst:
            cand = postprocess_vi(cand)
            if cand and cand not in seen:
                seen.add(cand)
                unique_candidates.append(cand)

    if not unique_candidates:
        return "", []

    top1_list = [lst[0] if lst else "" for lst in candidate_lists]
    total_weight = max(1e-9, sum(weights))

    lm_scores_raw = {}
    if target_lm is not None:
        for cand in unique_candidates:
            lm_scores_raw[cand] = target_lm.score(cand)
    lm_scores_norm = minmax_scale(lm_scores_raw) if lm_scores_raw else {cand: 0.0 for cand in unique_candidates}

    rows = []
    for cand in unique_candidates:
        consensus = 0.0
        for pred, w in zip(top1_list, weights):
            if pred:
                consensus += w * sentence_chrf(cand, pred)
        consensus /= total_weight

        presence = 0.0
        for lst, w in zip(candidate_lists, weights):
            if cand in lst:
                presence += w
        presence /= total_weight

        score = (
            GLOBAL_CFG.ensemble_consensus_weight * consensus
            + GLOBAL_CFG.ensemble_presence_weight * presence
            + GLOBAL_CFG.ensemble_lm_weight * lm_scores_norm.get(cand, 0.0)
        )

        rows.append(
            {
                "candidate": cand,
                "consensus": consensus,
                "presence": presence,
                "lm_norm": lm_scores_norm.get(cand, 0.0),
                "final_score": score,
            }
        )

    rows = sorted(rows, key=lambda x: x["final_score"], reverse=True)
    return rows[0]["candidate"], rows

def ensemble_decode_text(
    src_text: str,
    bundles: List[InferenceBundle],
    target_lm: Optional[SimpleWordNGramLM] = None,
    return_debug: bool = False,
):
    candidate_lists = []
    weights = []
    per_model = []

    for bundle in bundles:
        cands = beam_search_decode_nbest(
            model=bundle.model,
            src_text=src_text,
            sp_src=bundle.sp_src,
            sp_tgt=bundle.sp_tgt,
            special=bundle.special,
            exp=bundle.exp,
            nbest=GLOBAL_CFG.nbest_per_model,
        )
        candidate_lists.append(cands)
        weights.append(bundle.weight)
        per_model.append({"model": bundle.name, "weight": bundle.weight, "candidates": cands})

    winner, scored = choose_candidate_from_lists(candidate_lists, weights, target_lm=target_lm)

    if return_debug:
        return winner, {"src": src_text, "per_model": per_model, "scored_candidates": scored}
    return winner

def ensemble_predict_texts(
    src_texts: List[str],
    bundles: List[InferenceBundle],
    target_lm: Optional[SimpleWordNGramLM] = None,
    desc: str = "ensemble decode",
    return_debug: bool = False,
):
    preds = []
    debug_rows = []
    for src_text in tqdm(src_texts, desc=desc):
        if return_debug:
            pred, debug = ensemble_decode_text(src_text, bundles, target_lm=target_lm, return_debug=True)
            preds.append(pred)
            debug_rows.append(debug)
        else:
            preds.append(ensemble_decode_text(src_text, bundles, target_lm=target_lm, return_debug=False))
    return (preds, debug_rows) if return_debug else preds


## Fit the target LM and preview ensemble quality on the monitor-valid split

In [16]:
target_lm = None
if GLOBAL_CFG.use_target_lm_rerank:
    target_lm = SimpleWordNGramLM(order=3, add_k=0.2).fit(clean_train_vi)

preview_n = min(GLOBAL_CFG.preview_valid_samples, len(mon_va_src))
preview_preds, preview_debug = ensemble_predict_texts(
    mon_va_src[:preview_n],
    bundles=bundles,
    target_lm=target_lm,
    desc="ensemble preview",
    return_debug=True,
)

preview_bleu = corpus_bleu_score(preview_preds, mon_va_tgt[:preview_n])
preview_chrf = corpus_chrf_score(preview_preds, mon_va_tgt[:preview_n])

print(f"Preview ensemble BLEU on first {preview_n} valid rows: {preview_bleu:.4f}")
print(f"Preview ensemble chrF on first {preview_n} valid rows: {preview_chrf:.4f}")

preview_df = pd.DataFrame(
    {
        "zh": mon_va_src[: min(15, preview_n)],
        "reference_vi": mon_va_tgt[: min(15, preview_n)],
        "pred_vi": preview_preds[: min(15, preview_n)],
    }
)
preview_df


ensemble preview:   0%|          | 0/120 [00:00<?, ?it/s]/home/izu/Projects/.venv/.venv-310/lib/python3.10/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
ensemble preview: 100%|██████████| 120/120 [01:17<00:00,  1.54it/s]

Preview ensemble BLEU on first 120 valid rows: 49.4404
Preview ensemble chrF on first 120 valid rows: 60.2056


,zh,reference_vi,pred_vi
0,十月 十日 的 健康 体育节 是 为了 促进 人民 的 健康 而 设立 的 法定 节假日 。,Ngày Sức khỏe - Thể_Thao ngày 10 tháng 10 là n...,"Ngày 10 tháng 1, ngày 11 tháng 3 là người nhận..."
1,联合 航空 公司 801 次 航班 在 哪 取 行李 ?,Nơi lấy hành_lý của chuyến bay UA_801 ở đâu?,Chuyến bay JAL 801 ở đâu?
2,现在 最 流行 的 电影 是 什么 ?,Bộ phim phổ_biến nhất hiện_nay là gì?,Bộ phim phổ_biến nhất hiện_nay là gì?
3,稍后 我们 会 告诉 修理工 。,Chúng_tôi sẽ cử người sửa_chữa đến sau.,Chúng_tôi sẽ cho tôi biết sửa_chữa sau.
4,你 是 什么 血型 ?,Nhóm máu của bạn là gì?,Tên của bạn là gì?
5,在 五点 十分 起飞 。,Nó sẽ cất_cánh vào 17: 10.,Nó cất_cánh vào lúc 5 giờ 10 phút.
6,粤菜 跟 川菜 味道 不 一样 。,Hương_vị món ăn Quảng_Đông khác với món ăn Tứ_...,Thức_ăn ngon với món tráng_miệng.
7,有 朋友 来 看 我 。,có bạn đến thăm tôi.,có bạn đến thăm tôi.
8,我 想 问问 你 第三 点 。,Tôi muốn hỏi bạn về điểm_số ba.,Tôi muốn hỏi bạn gọi thứ ba.
9,"我 找 不 到 你 的 预订 , 但是 我们 有 一 个 房间 。",Tôi không_thể tìm thấy phòng đặt trước của bạn...,"Tôi không_thể tìm thấy phòng của bạn, nhưng ch..."


## Full validation ensemble score (optional but recommended)

This can take a while because each sentence is decoded by multiple models.


In [17]:
ensemble_valid_preds = ensemble_predict_texts(
    mon_va_src,
    bundles=bundles,
    target_lm=target_lm,
    desc="ensemble full valid",
    return_debug=False,
)
ensemble_valid_bleu = corpus_bleu_score(ensemble_valid_preds, mon_va_tgt)
ensemble_valid_chrf = corpus_chrf_score(ensemble_valid_preds, mon_va_tgt)

print(f"Ensemble valid BLEU : {ensemble_valid_bleu:.4f}")
print(f"Ensemble valid chrF : {ensemble_valid_chrf:.4f}")


ensemble full valid:   0%|          | 0/994 [00:00<?, ?it/s]/home/izu/Projects/.venv/.venv-310/lib/python3.10/site-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
ensemble full valid: 100%|██████████| 994/994 [11:55<00:00,  1.39it/s]


Ensemble valid BLEU : 47.3980
Ensemble valid chrF : 58.6054


## Generate final OJ submission with ensemble voting

In [18]:
test_preds, test_debug = ensemble_predict_texts(
    clean_test_zh,
    bundles=bundles,
    target_lm=target_lm,
    desc="ensemble test decode",
    return_debug=True,
)

pred_dir = work_dir / "final_predictions"
pred_dir.mkdir(parents=True, exist_ok=True)

submission_df = pd.DataFrame(
    {
        "tieng_trung": raw_test_zh,   # keep original raw test lines in CSV
        "tieng_viet": [postprocess_vi(x) for x in test_preds],
    }
)

assert len(submission_df) == len(raw_test_zh)
assert submission_df["tieng_viet"].isna().sum() == 0

submission_csv = pred_dir / "submission_v3.csv"
submission_zip = pred_dir / "submission_v3.zip"
debug_jsonl = pred_dir / "submission_v3_debug.jsonl"

submission_df.to_csv(submission_csv, index=False, encoding="utf-8")

with zipfile.ZipFile(submission_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_csv, arcname="submission.csv")

with open(debug_jsonl, "w", encoding="utf-8") as f:
    for row in test_debug:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("saved:", submission_csv.resolve())
print("saved:", submission_zip.resolve())
print("saved:", debug_jsonl.resolve())

submission_df.head()


ensemble test decode: 100%|██████████| 6413/6413 [56:15<00:00,  1.90it/s]  


saved: /home/izu/Projects/olpai/trans/mt_v3_artifacts/final_predictions/submission_v3.csv
saved: /home/izu/Projects/olpai/trans/mt_v3_artifacts/final_predictions/submission_v3.zip
saved: /home/izu/Projects/olpai/trans/mt_v3_artifacts/final_predictions/submission_v3_debug.jsonl


,tieng_trung,tieng_viet
0,还 早 呀 ， 过 几 年 再 说 。,"còn sớm nữa, qua mấy năm nữa nói lại."
1,你 跟 我 去吧,bạn cùng tôi đi
2,我 都 喜欢 。,Tôi đều thích nó.
3,我 受伤 了 需要 一 辆 救护车 。,Tôi bị mất một chiếc xe_cứu_thương.
4,等 一下 。 这个 是 吗 ？,Xin vui_lòng chờ cái này.


## Practical notes for better leaderboard score

The biggest improvement knobs in this v3 notebook are now:

1. **Use 3-model ensemble**, not just 1 model
2. leave **`do_full_retrain=True`** so final models use all labeled training data
3. if VRAM allows, keep the `bpe_wide_s2024` run
4. increase `epochs` by **2-4** for the best-performing preset
5. raise `nbest_per_model` from **2 → 3** for stronger voting
6. try one more seed for the best tokenizer family, for example:
   - another `bpe_base` with seed `3407`
   - another `unigram_base` with seed `888`
7. if inference is too slow, reduce to the best **2** models first, then compare valid BLEU

Good next experiment if your score is still stuck:
- duplicate the best preset
- only change the seed
- ensemble the two best BPE runs + one unigram run
